# Training YOLO11: RED, GREEN, YELLOW untuk ICAM-300 + robot arm

Notebook ini memakai `arm-warna-3.v5i.yolov11.zip` dan protokol serial **`v4.ino`**.
Disiapkan untuk training di **Google Colab berlangganan + NVIDIA A100**;
salin bundle hasil ekspor ke ICAM-300. A100 dipakai untuk training, ICAM untuk inference.
Default model: **`yolo11n-seg.pt`**, karena dataset berisi **poligon instance segmentation**.
Output: kelas warna, confidence, kontur, pusat mask dalam piksel, dan pratinjau koordinat `XY` robot.

| ID dataset (dipertahankan) | Nama asal | Nama model |
|---|---|---|
| 0 | HIJAU | GREEN |
| 1 | KUNING | YELLOW |
| 2 | MERAH | RED |

**Temuan pada ZIP yang diberikan:** train 3.480, valid 233, test 94 gambar; total 3.807.
Setiap gambar memiliki satu anotasi; semua bounding box poligon mencakup >80% luas gambar.
Sampel tampak berupa crop objek yang diperbesar. Angka mAP dari split ini **belum membuktikan**
kemampuan melokalisasi banyak benda kecil di meja pada foto Anda.
Tambahkan foto **frame penuh ICAM-300** dengan beberapa objek, background kosong/negatif,
berbagai posisi dan ukuran, lalu anotasi tiap instance RED/GREEN/YELLOW.
Pisahkan train/valid/test berdasarkan sesi pengambilan sebelum augmentasi agar gambar berdekatan tidak bocor antar split.

ICAM-300 menggunakan **Qualcomm QCS6490**. Runtime di bawah memakai **ONNX Runtime CPU**
dan CAMNavi2 dari BSP Advantech; kecepatan harus diukur di perangkat. Ekspor ONNX tidak otomatis
mengaktifkan DSP/NPU. Integrasi QNN/SNPE/QIM memerlukan konversi dan postprocessing yang cocok
dengan versi SDK serta output segmentation model ini.

Firmware `v4.ino` mengendalikan 5 axis stepper (J2/J3 berbagi satu motor) + servo gripper.
Integrasi mengikuti kalibrasi `SAVE:1..4` dan `XY:x,y`, bukan asumsi inverse kinematics 6 joint independen.
Model vision tidak menghasilkan sudut motor, kedalaman, atau trajektori pick-and-place.

## 1. Persiapan Google Colab A100

1. Upload notebook ini ke Google Colab.
2. Pilih **Runtime → Change runtime type → A100 GPU**, lalu hubungkan runtime.
3. Pada cell **Pilih/upload dataset ZIP**, pilih file ZIP dari komputer saat tombol upload muncul.
   Alternatif: simpan ZIP pada `MyDrive/arm-color-yolo/datasets/arm-warna-3.v5i.yolov11.zip`,
   atau isi `ZIP_SOURCE` dengan path ZIP yang sudah ada di Drive.
4. Jalankan cell berurutan. Cell penyimpanan meminta akses Drive melalui UI Colab.

Default A100: **batch 64, image 512, 100 epoch, AMP aktif**; workers mengikuti CPU runtime (maksimal 4).
GPU yang terpasang diperiksa oleh kode. Model tetap YOLO11n-seg agar deployment ICAM ringan.
Jika mengganti model/ukuran gambar dan mengalami CUDA out-of-memory, turunkan `BATCH` ke 32 atau 16.
Tidak perlu API key Roboflow. Instalasi berikut khusus mesin training dan memakai PyTorch bawaan Colab.
Dependensi training mendukung **Python 3.10–3.13**, termasuk Python 3.13 pada log Colab Anda:
NumPy 2.2.6 dan ONNX 1.18.0 memiliki wheel Python 3.13. `--only-binary=:all:` mencegah
pip mencoba kompilasi source. Pin lama NumPy 1.26.4 / ONNX 1.17.0 tidak dipakai untuk Colab.

**Setelah cell instalasi berhasil, pilih Runtime → Restart session satu kali**, lalu lanjutkan
dari cell pemeriksaan environment. Ini membersihkan modul versi lama yang mungkin sudah dimuat.
Tidak perlu menjalankan ulang instalasi setelah restart. Jangan pilih Disconnect and delete runtime
karena file `/content` akan hilang. Jika ada pesan `ERROR`, jangan lanjut ke training.

In [ ]:
%pip install --only-binary=:all: "ultralytics==8.3.221" "numpy==2.2.6" "opencv-python==4.11.0.86" "PyYAML==6.0.2" "onnx==1.18.0" "onnxruntime==1.20.1" "pyserial==3.5"

# Setelah instalasi berhasil, restart sekali melalui Runtime > Restart session.
# Setelah tersambung kembali, lanjutkan ke pemeriksaan environment; jangan ulangi instalasi.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 131.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 142.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: PyYAML
    Found existing installation: PyYAML 6.0.3
    Uninstalling PyYAML-6.0.3:
      Successfully uninstalled PyYAML-6.0.3
  Attempting uninstall: numpy
    Found existing install

In [ ]:
# Jalankan setelah Restart session. Cek import/ABI sebelum memakai waktu A100 untuk training.
import sys
from importlib.metadata import version

print('Python:', sys.version.split()[0])
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('Dependensi notebook ini disiapkan untuk Python 3.10–3.13.')
for package in ('numpy', 'onnx', 'onnxruntime'):
    loaded = sys.modules.get(package)
    if loaded is not None and getattr(loaded, '__version__', None) != version(package):
        raise RuntimeError(f'{package} versi lama masih dimuat. Runtime > Restart session, lalu ulangi cell ini.')

import numpy as np
import cv2
import onnx
import onnxruntime as ort
import torch
from ultralytics import YOLO

for package in ('ultralytics', 'numpy', 'opencv-python', 'onnx', 'onnxruntime', 'torch', 'torchvision'):
    print(package, version(package))

# Model Identity kecil: memverifikasi bahwa NumPy, ONNX, ORT dan PyTorch dapat bertukar tensor.
probe = np.array([[1.0, 2.0, 3.0]], dtype=np.float32)
graph = onnx.helper.make_graph(
    [onnx.helper.make_node('Identity', ['x'], ['y'])], 'environment_check',
    [onnx.helper.make_tensor_value_info('x', onnx.TensorProto.FLOAT, [1, 3])],
    [onnx.helper.make_tensor_value_info('y', onnx.TensorProto.FLOAT, [1, 3])],
)
probe_model = onnx.helper.make_model(graph, opset_imports=[onnx.helper.make_opsetid('', 13)], ir_version=8)
onnx.checker.check_model(probe_model)
probe_session = ort.InferenceSession(probe_model.SerializeToString(), providers=['CPUExecutionProvider'])
np.testing.assert_allclose(probe_session.run(None, {'x': probe})[0], probe)
np.testing.assert_allclose(torch.from_numpy(probe).numpy(), probe)
assert cv2.resize(np.zeros((4, 4, 3), dtype=np.uint8), (2, 2)).shape == (2, 2, 3)
del probe_session, probe_model, graph, probe
print('Environment OK. Lanjutkan ke mount Drive dan konfigurasi A100.')

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_GOOGLE_DRIVE = True
ROOT = Path('/content') if IN_COLAB else Path.cwd()
os.chdir(ROOT)
WORK = ROOT/'arm_color_work'  # Ekstraksi gambar tetap pada disk lokal runtime.
WORK.mkdir(parents=True, exist_ok=True)
if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')
    PERSIST = Path('/content/drive/MyDrive/arm-color-yolo')
else:
    PERSIST = WORK
    print('Drive tidak aktif: unduh hasil sebelum runtime berakhir.')
RUNS_ROOT = PERSIST/'runs'
ARTIFACT_ROOT = PERSIST/'artifacts'
for folder in (PERSIST/'datasets', RUNS_ROOT, ARTIFACT_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
print('Checkpoint:', RUNS_ROOT)
print('Model/bundle:', ARTIFACT_ROOT)

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import math
import os
import shutil
import warnings
import zipfile
import cv2
import numpy as np
import yaml
import torch
from ultralytics import YOLO
from IPython.display import display, Image

ZIP_NAME = 'arm-warna-3.v5i.yolov11.zip'
MODEL = 'yolo11n-seg.pt'
IMGSZ = 512  # Dataset Roboflow: stretch 512x512.
EPOCHS = 100
REQUIRE_GPU = IN_COLAB
HAS_CUDA = torch.cuda.is_available()
if REQUIRE_GPU and not HAS_CUDA:
    raise RuntimeError('GPU belum aktif. Pilih Runtime > Change runtime type > A100 GPU.')
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else 'CPU'
VRAM_GB = torch.cuda.get_device_properties(0).total_memory/1024**3 if HAS_CUDA else 0
BATCH = 64 if 'A100' in GPU_NAME.upper() and VRAM_GB >= 32 else (16 if HAS_CUDA else 4)
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
WORKERS = min(4, os.cpu_count() or 1) if IN_COLAB else 0
SEED = 42
CONF = 0.60  # Sesuaikan dari precision/recall valid + rekaman meja penuh.
RESUME_CHECKPOINT = None  # Contoh: RUNS_ROOT/'arm_colors/weights/last.pt' dari sesi terputus.
print(f'GPU: {GPU_NAME} | VRAM: {VRAM_GB:.1f} GiB | batch: {BATCH} | workers: {WORKERS}')
print('Device:', DEVICE, '| dataset dipilih pada cell berikutnya.')
if HAS_CUDA and 'A100' not in GPU_NAME.upper():
    print('GPU yang tersedia bukan A100; batch 16 dipakai. Ubah jenis runtime jika ingin A100.')
if DEVICE == 'cpu':
    print('Training CPU bisa lama. Untuk cek alur dulu, set EPOCHS=1; training final gunakan GPU.')

### Pilih/upload dataset ZIP

File pada drive Windows `R:` belum tersedia di mesin Colab. Cell ini mencari ZIP di `/content`
dan folder `datasets` pada Drive. Jika belum ditemukan, Colab menampilkan tombol upload:
pilih **`arm-warna-3.v5i.yolov11.zip` dari komputer Anda**, lalu tunggu upload selesai.

Jika ZIP sudah ada di folder Drive lain atau namanya berbeda, isi `ZIP_SOURCE` dengan path tersebut.
Contoh: `/content/drive/MyDrive/Dataset Robot/arm-warna-3.v5i.yolov11.zip`.
Path dapat disalin dari panel Files Colab. Nama hasil upload seperti `... (1).zip` juga diterima.
Untuk sesi berikutnya, simpan ZIP di folder Drive yang disebutkan di atas; upload ke `/content`
hanya tersimpan selama runtime tersedia. Setelah upload berhasil, lanjutkan ke audit dataset;
tidak perlu mengulang instalasi atau restart sesi.

In [ ]:
def resolve_dataset_zip(root, persist, source=None, upload=None):
    root, persist = Path(root), Path(persist)
    expected_name = 'arm-warna-3.v5i.yolov11.zip'
    if source is not None:
        selected = Path(source).expanduser()
        if not selected.is_absolute():
            selected = root/selected
    else:
        candidates = [root/expected_name, persist/'datasets'/expected_name]
        selected = next((path for path in candidates if path.is_file()), None)
        if selected is None and upload is not None:
            print('Pilih satu ZIP dataset dari komputer, lalu tunggu upload selesai.')
            uploaded = upload(target_dir=str(root))
            if len(uploaded) != 1:
                raise ValueError('Pilih tepat satu ZIP dataset. Jalankan ulang cell ini jika upload dibatalkan.')
            # files.upload mengembalikan nama file yang benar-benar disimpan, termasuk suffix (1).
            selected = Path(next(iter(uploaded)))
            del uploaded  # Lepaskan salinan bytes ZIP dari RAM setelah Colab menyimpannya.
            if not selected.is_absolute():
                selected = root/selected
        if selected is None:
            raise FileNotFoundError(f'Isi ZIP_SOURCE atau letakkan ZIP di {candidates[0]} atau {candidates[1]}.')
    if not selected.is_file():
        raise FileNotFoundError(f'ZIP_SOURCE tidak ditemukan: {selected}. Periksa path dari panel Files.')
    if selected.suffix.lower() != '.zip' or not zipfile.is_zipfile(selected):
        raise ValueError(f'File bukan arsip ZIP yang valid: {selected.name}')
    if selected.parent.resolve() != root.resolve():
        # Salin arsip Drive satu kali ke disk Colab, tanpa menimpa ZIP asli di /content.
        digest = hashlib.sha256(selected.read_bytes()).hexdigest()[:12]
        local = root/'arm_color_work'/'archives'/digest/selected.name
        local.parent.mkdir(parents=True, exist_ok=True)
        if selected.resolve() != local.resolve():
            shutil.copy2(selected, local)
        selected = local
    return selected.resolve()


ZIP_SOURCE = None  # Atau isi path ZIP yang sudah ada di Drive / folder lokal.
upload_zip = None
if IN_COLAB:
    from google.colab import files
    upload_zip = files.upload
ZIP_PATH = resolve_dataset_zip(ROOT, PERSIST, source=ZIP_SOURCE, upload=upload_zip)
print(f'Dataset siap: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024**2:.1f} MiB)')

## 2. Ekstrak, perbaiki YAML, dan audit anotasi

ZIP mempunyai tiga entri `data.yaml` identik. Ekstraksi di bawah menerima duplikat identik,
menolak duplikat yang bertentangan, dan menulis ke folder berdasarkan hash ZIP sehingga
dataset versi berbeda tidak tercampur. Nama kelas diganti **tanpa mengubah ID label**.
Path `../train/images` dari ekspor diperbaiki menjadi path dataset yang sebenarnya.

In [ ]:
def prepare_dataset(zip_path, work):
    digest = hashlib.sha256(Path(zip_path).read_bytes()).hexdigest()
    target = Path(work) / ('dataset_' + digest[:12])
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        seen = {}
        for info in archive.infolist():
            dest = (target / info.filename).resolve()
            if not dest.is_relative_to(target.resolve()):
                raise ValueError('Path ZIP keluar dari folder dataset')
            if info.is_dir():
                dest.mkdir(parents=True, exist_ok=True)
                continue
            payload = archive.read(info)
            fingerprint = hashlib.sha256(payload).hexdigest()
            if info.filename in seen and seen[info.filename] != fingerprint:
                raise ValueError('Entri ZIP duplikat berbeda: ' + info.filename)
            if info.filename not in seen:
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(payload)
                seen[info.filename] = fingerprint
    yamls = list(target.rglob('data.yaml'))
    if len(yamls) != 1:
        raise ValueError('Harus ada satu lokasi data.yaml')
    base = yamls[0].parent
    original = yaml.safe_load(yamls[0].read_text())
    original_names = original['names']
    if isinstance(original_names, list):
        original_names = dict(enumerate(original_names))
    aliases = {'HIJAU': 'GREEN', 'KUNING': 'YELLOW', 'MERAH': 'RED',
               'GREEN': 'GREEN', 'YELLOW': 'YELLOW', 'RED': 'RED'}
    names = {int(k): aliases[str(v).strip().upper()] for k, v in original_names.items()}
    if sorted(names) != [0, 1, 2] or set(names.values()) != {'RED', 'GREEN', 'YELLOW'}:
        raise ValueError('Dataset harus tepat tiga kelas warna')
    fixed = dict(path=str(base.resolve()), train='train/images', val='valid/images',
                 test='test/images', nc=3, names=names)
    if 'roboflow' in original:
        fixed['roboflow'] = original['roboflow']  # Pertahankan atribusi dataset.
    output = base / 'data_colors.yaml'
    output.write_text(yaml.safe_dump(fixed, sort_keys=False), encoding='utf-8')
    return base, output, names, digest


def audit_dataset(base, names):
    report, all_hashes, split_sources = {}, {}, {}
    for split in ('train', 'valid', 'test'):
        images = sorted(p for p in (base/split/'images').iterdir()
                        if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})
        labels = {p.stem: p for p in (base/split/'labels').glob('*.txt')}
        if not images or len({p.stem for p in images}) != len(images):
            raise ValueError(f'{split}: gambar kosong atau nama stem duplikat')
        if set(labels) != {p.stem for p in images}:
            raise ValueError(f'{split}: pasangan image/label tidak lengkap (negatif gunakan label kosong)')
        counts, instances, large, total, sources = Counter(), [], 0, 0, set()
        for image_path in images:
            image = cv2.imread(str(image_path))
            if image is None:
                raise ValueError(f'Gambar rusak: {image_path}')
            fingerprint = hashlib.sha256(image_path.read_bytes()).hexdigest()
            all_hashes.setdefault(fingerprint, set()).add(split)
            sources.add(image_path.stem.split('.rf.')[0])
            rows = labels[image_path.stem].read_text().splitlines()
            instance_count = 0
            for row in rows:
                if not row.strip():
                    continue
                values = np.asarray([float(v) for v in row.split()])
                if len(values) < 7 or (len(values)-1) % 2 or not np.isfinite(values).all():
                    raise ValueError(f'Butuh poligon YOLO, bukan bbox/classification: {image_path.name}')
                cid = int(values[0])
                if values[0] != cid or cid not in names:
                    raise ValueError('Class ID tidak valid')
                points = values[1:].reshape(-1, 2)
                if (points < 0).any() or (points > 1).any() or abs(cv2.contourArea(points.astype(np.float32))) <= 1e-8:
                    raise ValueError(f'Poligon tidak valid: {image_path.name}')
                counts[names[cid]] += 1
                instance_count += 1
                large += int(np.prod(np.ptp(points, axis=0)) > 0.8)
                total += 1
            instances.append(instance_count)
        if set(counts) != set(names.values()):
            raise ValueError(f'{split}: ada kelas yang tidak punya contoh')
        split_sources[split] = sources
        report[split] = dict(images=len(images), instances=total, per_class=dict(counts),
                             negative_images=instances.count(0), multi_object_images=sum(n>1 for n in instances),
                             large_bbox_fraction=large/max(total, 1))
    exact_overlap = sum(len(v)>1 for v in all_hashes.values())
    source_overlap = {f'{a}/{b}': len(split_sources[a] & split_sources[b])
                      for a, b in [('train', 'valid'), ('train', 'test'), ('valid', 'test')]}
    report['leakage'] = dict(exact_image_hashes=exact_overlap, source_name_overlap=source_overlap)
    if exact_overlap or any(source_overlap.values()):
        warnings.warn('Kemungkinan kebocoran split: pisahkan foto asal/sesi sebelum menilai mAP.')
    if report['train']['large_bbox_fraction'] > 0.8:
        warnings.warn('Dataset didominasi crop besar. Tambahkan scene meja penuh untuk lokalisasi multiobjek.')
    return report

DATASET, DATA_YAML, NAMES, ZIP_SHA256 = prepare_dataset(ZIP_PATH, WORK)
REPORT = audit_dataset(DATASET, NAMES)
(WORK/'dataset_audit.json').write_text(json.dumps(REPORT, indent=2))
print('Class mapping:', NAMES)
print(json.dumps(REPORT, indent=2))

In [ ]:
# Lihat satu contoh per kelas beserta poligon; warna OpenCV menggunakan BGR.
import matplotlib.pyplot as plt
palette = {'RED': (255, 0, 0), 'GREEN': (0, 180, 0), 'YELLOW': (220, 180, 0)}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for class_id, ax in enumerate(axes):
    for label in sorted((DATASET/'valid/labels').glob('*.txt')):
        rows = [list(map(float, row.split())) for row in label.read_text().splitlines() if row.strip()]
        if not any(int(row[0]) == class_id for row in rows):
            continue
        image_path = next(p for p in (DATASET/'valid/images').glob(label.stem+'.*'))
        frame = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
        h, w = frame.shape[:2]
        for row in rows:
            polygon = (np.array(row[1:]).reshape(-1, 2)*[w, h]).astype(np.int32)
            cv2.polylines(frame, [polygon], True, palette[NAMES[int(row[0])]], 2)
        ax.imshow(frame); ax.set_title(NAMES[class_id]); ax.axis('off')
        break
plt.show()

## 3. Train dan evaluasi

Hue dan pertukaran RGB/BGR dimatikan agar label warna tidak berubah akibat augmentasi.
Saturasi dimatikan, perubahan brightness dibatasi. Dataset sudah diaugmentasi Roboflow;
augmentasi tambahan dibuat ringan. Jangan menerapkan grayscale atau recolor tanpa mengganti label.
Mosaic/copy-paste sintetis tidak menggantikan foto meja nyata.

Model awal akan diunduh saat pertama dipakai. CPU/GPU dan batch dipilih di konfigurasi.
Di Colab workers mengikuti jumlah CPU (maksimal 4); di Windows digunakan 0.
`last.pt` disimpan setiap epoch dan `best.pt` saat metrik membaik ke Drive jika mount aktif;
snapshot tambahan disimpan setiap 10 epoch.
Dataset/cache tetap di `/content` agar training tidak membaca ribuan gambar dari Drive.

Untuk melanjutkan sesi yang terputus: sambungkan A100 kembali, jalankan instalasi, storage,
konfigurasi, dan audit dataset; isi `RESUME_CHECKPOINT` dengan path `last.pt` yang belum selesai,
lalu jalankan cell training. Resume memulihkan optimizer/epoch dan parameter run sebelumnya.
Gunakan path dataset `/content` yang sama. Checkpoint dari training yang sudah selesai bukan
untuk `resume=True`; biarkan `RESUME_CHECKPOINT=None` untuk memulai run baru.

In [ ]:
if RESUME_CHECKPOINT is not None:
    if not Path(RESUME_CHECKPOINT).is_file():
        raise FileNotFoundError(RESUME_CHECKPOINT)
    model = YOLO(str(RESUME_CHECKPOINT))
    train_result = model.train(resume=True, device=DEVICE)
else:
    model = YOLO(MODEL)
    train_result = model.train(
        data=str(DATA_YAML), task='segment', imgsz=IMGSZ, epochs=EPOCHS,
        batch=BATCH, device=DEVICE, workers=WORKERS, seed=SEED, deterministic=True,
        project=str(RUNS_ROOT), name='arm_colors', exist_ok=False,
        save=True, save_period=10, patience=20, cache=False, amp=(DEVICE != 'cpu'),
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.10, bgr=0.0,
        degrees=10, translate=0.05, scale=0.15, shear=0.0, perspective=0.0,
        fliplr=0.5, flipud=0.0, mosaic=0.0, mixup=0.0, copy_paste=0.0,
        plots=True,
    )
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR/'weights/best.pt'
assert BEST_PT.is_file(), 'best.pt tidak ditemukan; periksa hasil training'
print('Checkpoint:', BEST_PT)

In [ ]:
best = YOLO(str(BEST_PT))
assert best.names == NAMES
VALIDATION = {}
for split in ('val', 'test'):
    metrics = best.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ,
                       batch=BATCH, device=DEVICE, workers=WORKERS, plots=True,
                       project=str(PERSIST/'evaluation'), name=split, exist_ok=False)
    VALIDATION[split] = dict(box_map50=float(metrics.box.map50), box_map50_95=float(metrics.box.map),
                             mask_map50=float(metrics.seg.map50), mask_map50_95=float(metrics.seg.map),
                             per_class_mask_map50_95={NAMES[i]: float(v) for i, v in enumerate(metrics.seg.maps)})
print(json.dumps(VALIDATION, indent=2))
for filename in ('results.png', 'confusion_matrix.png', 'BoxPR_curve.png', 'MaskPR_curve.png'):
    path = RUN_DIR/filename
    if path.exists():
        display(Image(filename=str(path)))

## 4. Ekspor ONNX FP32 statis

Ekspor `batch=1`, `imgsz=512`, `nms=False`, `opset=13`; runtime melakukan NMS per kelas
dan rekonstruksi mask. Input runtime memakai **stretch resize** agar sesuai preprocessing ZIP;
koordinat dikembalikan ke resolusi frame asli dengan skala X/Y terpisah.
Jika dataset baru memakai letterbox, ubah preprocessing dan pemetaan balik runtime bersama-sama.

In [ ]:
import onnx
import onnxruntime as ort

ONNX_PATH = Path(best.export(format='onnx', imgsz=IMGSZ, batch=1, dynamic=False,
                             half=False, simplify=False, nms=False, opset=13, device='cpu'))
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))
session = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
print('Input:', [(t.name, t.shape, t.type) for t in session.get_inputs()])
print('Output:', [(t.name, t.shape) for t in session.get_outputs()])

# Bandingkan output raw PT dan ONNX pada gambar yang sama, sebelum NMS.
sample_path = next((DATASET/'test/images').glob('*.jpg'))
sample = cv2.imread(str(sample_path))
rgb = cv2.cvtColor(cv2.resize(sample, (IMGSZ, IMGSZ)), cv2.COLOR_BGR2RGB)
tensor = np.ascontiguousarray(rgb.transpose(2, 0, 1)[None], dtype=np.float32)/255
pt = YOLO(str(BEST_PT)).model.cpu().eval()
with torch.no_grad():
    pt_prediction = pt(torch.from_numpy(tensor))[0].numpy()
ort_outputs = session.run(None, {session.get_inputs()[0].name: tensor})
onnx_prediction = next(output for output in ort_outputs if output.ndim == 3)
np.testing.assert_allclose(pt_prediction, onnx_prediction, rtol=1e-3, atol=1e-3)
print('PT/ONNX raw prediction cocok pada satu gambar; lanjutkan uji rekaman ICAM penuh.')

## 5. Runtime ringan yang ikut dalam bundle

Cell berikut menulis `icam_color_runtime.py` ke folder kerja. Runtime tidak membutuhkan PyTorch/Ultralytics.
Itu menyediakan decoder segmentation, CAMNavi2, homography, dan adapter serial `v4.ino`.
CLI hanya melakukan deteksi dan mencetak pratinjau perintah; **tidak membuka port robot**.

In [ ]:
%%writefile icam_color_runtime.py
"""YOLO11-seg ONNX CPU / CAMNavi2 / serial v4.ino. No automatic robot motion."""
import argparse
import ast
import json
import queue
import re
import time
from pathlib import Path

import cv2
import numpy as np

COLORS = {"RED": (0, 0, 255), "GREEN": (0, 200, 0), "YELLOW": (0, 255, 255)}


def decode(prediction, prototypes, frame_shape, input_hw, names, conf=0.6, iou=0.45):
    """Decode raw YOLO11-seg, batch=1, nms=False; input image uses stretch resize."""
    if not 0 < conf <= 1 or not 0 < iou <= 1:
        raise ValueError("conf/iou harus 0 < nilai <= 1")
    if prediction.ndim != 3 or prototypes.ndim != 4 or prediction.shape[0] != 1 or prototypes.shape[0] != 1:
        raise ValueError("Butuh output YOLO11-seg raw, batch=1")
    nc, nm = len(names), prototypes.shape[1]
    if prediction.shape[1] != 4 + nc + nm:
        raise ValueError("Layout output harus [1, 4+nc+nm, anchors]; ekspor nms=False")
    rows = prediction[0].T
    rows = rows[np.isfinite(rows).all(axis=1)]
    cls = rows[:, 4:4+nc].argmax(axis=1)
    scores = rows[np.arange(len(rows)), 4+cls]
    keep = (scores >= conf) & (rows[:, 2] > 0) & (rows[:, 3] > 0)
    rows, cls, scores = rows[keep], cls[keep], scores[keep]
    boxes = np.column_stack((rows[:, :2] - rows[:, 2:4]/2, rows[:, 2:4]))
    selected = []
    for class_id in np.unique(cls):
        indices = np.flatnonzero(cls == class_id)
        local = cv2.dnn.NMSBoxes(boxes[indices].tolist(), scores[indices].tolist(), conf, iou)
        selected.extend(indices[np.asarray(local, dtype=int).reshape(-1)].tolist())
    h, w = frame_shape[:2]
    ih, iw = input_hw
    proto = prototypes[0]
    detections = []
    for index in sorted(selected, key=lambda k: -scores[k])[:20]:
        x, y, bw, bh = boxes[index]
        x1, x2 = np.clip([x*w/iw, (x+bw)*w/iw], 0, w).astype(int)
        y1, y2 = np.clip([y*h/ih, (y+bh)*h/ih], 0, h).astype(int)
        if x2 <= x1 or y2 <= y1:
            continue
        logits = (rows[index, 4+nc:] @ proto.reshape(nm, -1)).reshape(proto.shape[1:])
        mask = (cv2.resize(logits, (w, h))[y1:y2, x1:x2] > 0).astype(np.uint8)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea)
        moments = cv2.moments(contour)
        if moments['m00'] <= 0:
            continue
        center = [x1 + moments['m10']/moments['m00'], y1 + moments['m01']/moments['m00']]
        # An irregular mask centroid can lie outside the object: do not offer it for motion.
        center_inside = cv2.pointPolygonTest(contour, (center[0]-x1, center[1]-y1), False) >= 0
        detections.append(dict(class_id=int(cls[index]), color=names[int(cls[index])],
                               confidence=float(scores[index]), center_px=center,
                               center_inside_mask=center_inside, bbox_xyxy=[int(x1), int(y1), int(x2), int(y2)],
                               contour=(contour.reshape(-1, 2)+[x1, y1]).tolist()))
    return detections


class ColorModel:
    def __init__(self, model_path):
        import onnxruntime as ort
        self.session = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
        tensor = self.session.get_inputs()[0]
        if tensor.type != 'tensor(float)' or len(tensor.shape) != 4 or tensor.shape[:2] != [1, 3]:
            raise ValueError('Ekspor ONNX FP32 NCHW batch=1 diperlukan')
        if not all(isinstance(v, int) and v > 0 for v in tensor.shape):
            raise ValueError('Ekspor dynamic=False diperlukan')
        self.input_name, self.hw = tensor.name, tuple(tensor.shape[2:])
        meta = self.session.get_modelmeta().custom_metadata_map
        names = ast.literal_eval(meta.get('names', '{}'))
        self.names = {int(k): str(v) for k, v in names.items()}
        if sorted(self.names) != [0, 1, 2] or set(self.names.values()) != set(COLORS):
            raise ValueError('Metadata names harus tepat GREEN, YELLOW, RED dengan ID 0..2')

    def predict(self, frame, conf=0.6):
        if frame is None or frame.ndim != 3 or frame.shape[2] != 3:
            raise ValueError('Frame harus BGR 3 channel; kamera mono tidak dapat mendeteksi warna')
        rgb = cv2.cvtColor(cv2.resize(frame, (self.hw[1], self.hw[0])), cv2.COLOR_BGR2RGB)
        tensor = np.ascontiguousarray(rgb.transpose(2, 0, 1)[None], dtype=np.float32)/255.0
        outputs = self.session.run(None, {self.input_name: tensor})
        predictions = [v for v in outputs if v.ndim == 3]
        prototypes = [v for v in outputs if v.ndim == 4]
        if len(predictions) != 1 or len(prototypes) != 1:
            raise ValueError('Model harus YOLO11 instance segmentation dengan dua output raw')
        return decode(predictions[0], prototypes[0], frame.shape, self.hw, self.names, conf)


class PlaneCalibration:
    def __init__(self, config):
        self.size = tuple(config['frame_size_wh'])
        self.area = np.asarray(config['area_mm'], dtype=float)
        corners = np.asarray(config['corners_px'], dtype=np.float32)
        if len(self.size) != 2 or min(self.size) <= 0 or self.area.shape != (2,) or not np.isfinite(self.area).all() or (self.area <= 0).any():
            raise ValueError('Ukuran frame/area tidak valid')
        if corners.shape != (4, 2) or not np.isfinite(corners).all():
            raise ValueError('Isi 4 sudut piksel sesuai urutan SAVE:1,2,3,4')
        if (corners < 0).any() or (corners >= self.size).any():
            raise ValueError('Sudut di luar resolusi kalibrasi')
        if not cv2.isContourConvex(corners) or abs(cv2.contourArea(corners)) < 100:
            raise ValueError('Empat sudut harus membentuk segiempat cembung tanpa silang')
        self.corners = corners
        w, h = self.area
        self.matrix = cv2.getPerspectiveTransform(corners, np.float32([[0, 0], [w, 0], [w, h], [0, h]]))

    def xy(self, center_px, frame_shape):
        if (frame_shape[1], frame_shape[0]) != self.size:
            raise ValueError('Resolusi berubah: ulangi kalibrasi piksel')
        p = np.asarray(center_px, dtype=np.float32)
        if p.shape != (2,) or not np.isfinite(p).all():
            raise ValueError('Titik piksel tidak valid')
        if cv2.pointPolygonTest(self.corners, tuple(map(float, p)), False) < 0:
            return None
        out = cv2.perspectiveTransform(p.reshape(1, 1, 2), self.matrix)[0, 0]
        if not np.isfinite(out).all() or (out < -0.01).any() or (out > self.area+0.01).any():
            return None
        return np.clip(out, 0, self.area).tolist()


def xy_command(x, y, area_mm):
    values = np.asarray([x, y], dtype=float)
    bounds = np.asarray(area_mm, dtype=float)
    if bounds.shape != (2,) or not np.isfinite(bounds).all() or (bounds <= 0).any():
        raise ValueError('AREA tidak valid')
    if not np.isfinite(values).all() or (values < 0).any() or (values > bounds).any():
        raise ValueError('XY di luar AREA atau tidak finite')
    return f'XY:{x:.2f},{y:.2f}'


class V4Serial:
    """Single writer, robot idle before opening; no ZERO/HOME/SERVO is sent."""
    def __init__(self, port=None, area_mm=(300, 200), dry_run=True, timeout=120):
        xy_command(0, 0, area_mm)
        self.area, self.dry_run, self.timeout = area_mm, dry_run, timeout
        self.faulted, self.serial = False, None
        if not dry_run:
            import serial
            if not port:
                raise ValueError('Isi port USB serial ESP32')
            self.serial = serial.Serial(port, 115200, timeout=0.2, write_timeout=1)
            try:
                time.sleep(2)  # Opening a USB serial port may reset ESP32.
                self.serial.reset_input_buffer()
                self.serial.write(b'POINTS\n')
                lines, end = [], time.monotonic()+5
                while time.monotonic() < end:
                    line = self.serial.readline().decode(errors='replace').strip()
                    lines.append(line)
                    if line.startswith('Area W x H ='):
                        break
                for n in range(1, 5):
                    if not any(s.startswith(f'Titik {n} ') and s.endswith(' deg') for s in lines):
                        raise RuntimeError('Kalibrasi firmware belum lengkap atau firmware bukan v4.ino')
                match = re.search(r'Area W x H = ([\d.]+) x ([\d.]+) mm', '\n'.join(lines))
                if not match or not np.allclose([float(v) for v in match.groups()], area_mm, atol=0.01, rtol=0):
                    raise RuntimeError('AREA firmware berbeda dengan kalibrasi kamera')
            except BaseException:
                self.close()
                raise

    def move_xy(self, x, y):
        command = xy_command(x, y, self.area)
        if self.faulted:
            raise RuntimeError('Koneksi fault; periksa robot sebelum membuka sesi baru')
        if self.dry_run:
            print('DRY RUN:', command)
            return command
        try:
            self.serial.reset_input_buffer()
            self.serial.write((command+'\n').encode('ascii'))
            deadline = time.monotonic()+self.timeout
            accepted = False
            while time.monotonic() < deadline:
                line = self.serial.readline().decode(errors='replace').strip()
                if line.startswith(('LIMIT', 'ZERO belum', 'Kalibrasi belum', 'XY di luar', 'Format', 'Command tidak')):
                    raise RuntimeError(line)
                if line == 'STOP' or 'ESP32 ROBOT ARM' in line:
                    raise RuntimeError('STOP/reset terdeteksi saat menunggu gerakan')
                if line.startswith('XY ->'):
                    accepted = True
                if accepted and line in ('Motion selesai', 'Sudah di posisi target'):
                    return command
            raise TimeoutError('Tidak menerima Motion selesai / Sudah di posisi target')
        except BaseException:
            self.faulted = True
            try:
                self.serial.write(b'STOP\n')
            except Exception:
                pass
            raise

    def close(self):
        if self.serial is not None:
            self.serial.close()


class CamNaviSource:
    """CAMNavi2 JPEG callback, latest frame only. Requires vendor BSP SDK."""
    def __init__(self, device_name, width=640, height=480):
        from CamNavi2 import CamNavi2
        self.sdk = CamNavi2.CamNavi2() if hasattr(CamNavi2, 'CamNavi2') else CamNavi2()
        print('CAMNavi devices:', self.sdk.enum_camera_list())
        self.camera = self.sdk.get_device_by_name(device_name)
        self.frames = queue.Queue(maxsize=1)
        if self.camera is None:
            raise RuntimeError('Nama device tidak ditemukan; gunakan hasil enum_camera_list()')
        if int(self.sdk.advcam_query_fw_sku(self.camera)) != 1:
            raise RuntimeError('SDK tidak melaporkan kamera color (SKU=1); verifikasi BSP')
        try:
            self.sdk.advcam_config_pipeline(self.camera, acq_mode=0, width=width, height=height,
                                           enable_infer=0, pipeline_mode='default')
            self.sdk.advcam_open(self.camera)
            self.sdk.advcam_register_new_image_handler(self.camera, self._sample)
            self.sdk.advcam_play(self.camera)
        except BaseException:
            self.close()
            raise

    def _sample(self, sample):
        if sample is None:
            return
        buf = sample.get_buffer()
        frame = cv2.imdecode(np.frombuffer(buf.extract_dup(0, buf.get_size()), np.uint8), cv2.IMREAD_UNCHANGED)
        if frame is None or frame.ndim != 3 or frame.shape[2] != 3:
            return
        if self.frames.full():
            try:
                self.frames.get_nowait()
            except queue.Empty:
                pass
        try:
            self.frames.put_nowait((time.monotonic(), frame))
        except queue.Full:
            pass

    def read(self, timeout=5):
        deadline = time.monotonic()+timeout
        while time.monotonic() < deadline:
            try:
                stamp, frame = self.frames.get(timeout=max(0.001, deadline-time.monotonic()))
            except queue.Empty:
                break
            if time.monotonic()-stamp < 1:
                return frame
        raise TimeoutError('Tidak ada frame JPEG berwarna baru dari CAMNavi2')

    def close(self):
        try:
            self.sdk.advcam_register_new_image_handler(self.camera, None)
        finally:
            self.sdk.advcam_close(self.camera)


def annotate(frame, detections):
    out = frame.copy()
    for d in detections:
        color = COLORS[d['color']]
        cv2.polylines(out, [np.asarray(d['contour'], np.int32)], True, color, 2)
        p = tuple(np.round(d['center_px']).astype(int))
        cv2.circle(out, p, 4, color, -1)
        cv2.putText(out, f"{d['color']} {d['confidence']:.2f}", p,
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return out


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--model', default='best.onnx')
    parser.add_argument('--source', default='camnavi', help='camnavi, image path, video, RTSP, or webcam index')
    parser.add_argument('--device-name', default='iCam500', help='Use actual CAMNavi enum name; manual uses iCam500')
    parser.add_argument('--width', type=int, default=640)
    parser.add_argument('--height', type=int, default=480)
    parser.add_argument('--conf', type=float, default=0.6)
    parser.add_argument('--calibration', help='Optional calibrated camera JSON; outputs XY command previews only')
    parser.add_argument('--show', action='store_true', help='Requires GUI OpenCV and local display')
    parser.add_argument('--frames', type=int, default=100)
    args = parser.parse_args()
    model = ColorModel(args.model)
    calibration = PlaneCalibration(json.loads(Path(args.calibration).read_text())) if args.calibration else None
    source = None
    try:
        still = cv2.imread(args.source) if Path(args.source).suffix.lower() in {'.jpg', '.jpeg', '.png'} else None
        if args.source == 'camnavi':
            source = CamNaviSource(args.device_name, args.width, args.height)
        elif still is None:
            source = cv2.VideoCapture(int(args.source) if args.source.isdigit() else args.source)
            if not source.isOpened():
                raise RuntimeError('Sumber kamera/video tidak dapat dibuka')
        for _ in range(args.frames):
            start = time.monotonic()
            if still is not None:
                frame = still
            elif isinstance(source, CamNaviSource):
                frame = source.read()
            else:
                ok, frame = source.read()
                if not ok:
                    break
            detections = model.predict(frame, args.conf)
            for detection in detections:
                if calibration and detection['center_inside_mask']:
                    xy = calibration.xy(detection['center_px'], frame.shape)
                    detection['xy_mm'] = xy
                    detection['command_preview'] = xy_command(*xy, calibration.area) if xy is not None else None
            print(json.dumps(dict(detections=detections, elapsed_ms=round((time.monotonic()-start)*1000, 1))))
            overlay = annotate(frame, detections)
            if still is not None:
                cv2.imwrite('prediction.jpg', overlay)
                break
            if args.show:
                cv2.imshow('RED GREEN YELLOW', overlay)
                if cv2.waitKey(1) & 0xff in (27, ord('q')):
                    break
    finally:
        if isinstance(source, CamNaviSource):
            source.close()
        elif source is not None:
            source.release()
        if args.show:
            cv2.destroyAllWindows()


if __name__ == '__main__':
    main()

In [ ]:
from icam_color_runtime import ColorModel, PlaneCalibration, V4Serial, annotate, xy_command

detector = ColorModel(ONNX_PATH)
detections = detector.predict(sample, conf=CONF)
display(Image(data=cv2.imencode('.jpg', annotate(sample, detections))[1].tobytes()))
print([{k: v for k, v in d.items() if k != 'contour'} for d in detections])

# Isi dengan foto frame PENUH meja ICAM-300 untuk uji multiobjek yang bermakna.
SCENE_IMAGE = None  # Contoh: ROOT/'meja_3_warna.jpg'
if SCENE_IMAGE is not None:
    scene = cv2.imread(str(SCENE_IMAGE))
    found = detector.predict(scene, CONF)
    display(Image(data=cv2.imencode('.jpg', annotate(scene, found))[1].tobytes()))
    print(Counter(d['color'] for d in found))

## 6. Kalibrasi kamera ke `XY` firmware

Lakukan ini di komputer yang terhubung langsung ke ESP32/ICAM, bukan port USB Colab.
Pastikan firmware yang dipasang adalah **`v4.ino`**: environment PlatformIO default repo ini
masih membangun firmware MQTT di `src/`, yang protokol dan pinnya berbeda.

1. Atur pose referensi fisik, lalu `ZERO` melalui Serial Monitor 115200 baud saat robot diam.
   Nilai tersimpan tidak membuktikan pose fisik masih sama setelah listrik mati/step motor terlewat.
2. Tentukan ukuran bidang kerja sebenarnya: `AREA:W,H` dalam mm.
3. Jog joint ke empat sudut pada **ketinggian dan orientasi yang sama**, tunggu gerakan selesai,
   simpan `SAVE:1` (kiri-dekat robot, 0,0), `SAVE:2` (kanan-dekat, W,0),
   `SAVE:3` (kanan-jauh, W,H), `SAVE:4` (kiri-jauh, 0,H). Periksa `POINTS`.
4. Ambil frame kamera. Catat koordinat piksel **sudut fisik yang sama** dalam urutan SAVE:1..4;
   urutan tersebut tidak selalu kiri-atas gambar. Isi JSON di bawah dengan hasil pengukuran.
5. Kamera harus tetap, resolusi/crop tidak berubah, distorsi lensa kecil atau dikoreksi konsisten.
   Homography berlaku untuk satu bidang/ketinggian; perbedaan tinggi objek menimbulkan paralaks.
   Uji juga titik tengah: interpolasi joint bilinear firmware tidak menjamin lintasan/XY kartesian tepat.

Tidak ada angka piksel kalibrasi yang dapat dipercaya hanya dari foto ilustrasi.
Pusat mask hanya kandidat target, belum tentu titik genggam yang baik.
`XY:x,y,servo` sengaja tidak dipakai karena firmware menjalankan servo **sebelum** gerak selesai.
Gerak angkat, turun, genggam, dan drop-zone tiap warna harus diajarkan serta diuji terpisah.

In [ ]:
CALIBRATION = {
    'frame_size_wh': [640, 480],  # Ganti sesuai frame ICAM yang digunakan.
    'area_mm': [300.0, 200.0],   # Ganti sesuai AREA:W,H fisik.
    'corners_px': None,          # [[u1,v1],[u2,v2],[u3,v3],[u4,v4]], urutan SAVE:1..4.
}
calibration = None
if CALIBRATION['corners_px'] is not None:
    calibration = PlaneCalibration(CALIBRATION)
    (WORK/'camera_calibration.json').write_text(json.dumps(CALIBRATION, indent=2))
    print('Kalibrasi disimpan. Uji titik tengah dan beberapa titik dalam bidang sebelum gerak.')
else:
    print('Deteksi dapat berjalan; koordinat robot dinonaktifkan sampai corners_px diisi.')

# Simulasi protokol, tidak membuka port serial dan tidak memerlukan kalibrasi kamera.
robot_preview = V4Serial(dry_run=True, area_mm=CALIBRATION['area_mm'])
robot_preview.move_xy(CALIBRATION['area_mm'][0]/2, CALIBRATION['area_mm'][1]/2)

### Uji satu target dari frame kamera baru (opsional, setelah bundle dipasang di ICAM)

Cell ini sengaja dinonaktifkan saat Run All. Sesudah memastikan ZERO, empat titik,
pose robot diam, dan lintasan bebas, ubah flag untuk menguji **satu** target.
Tutup Serial Monitor sebelum membuka port ini. Adapter memeriksa `POINTS`/AREA dan menunggu
`Motion selesai` atau `Sudah di posisi target`; error/timeout memicu `STOP` dan sesi ditandai fault.
`STOP` pada firmware adalah perlambatan stepper, bukan pemutus daya darurat.
Tidak ada antrean gerak otomatis atau pemilihan kotak drop berdasarkan warna di firmware ini.

In [ ]:
SEND_ONE_TARGET = False
CALIBRATION_VERIFIED = False  # Set True hanya setelah uji pemetaan fisik di meja.
DEPLOY_DIR = '.'  # Folder hasil ekstraksi bundle di ICAM (best.onnx + camera_calibration.json).
SERIAL_PORT = '/dev/ttyUSB0'  # Di Windows: COMx; sesuaikan hasil enumerasi port.
CAMERA_DEVICE_NAME = 'iCam500'  # Sesuaikan enum_camera_list() pada BSP Anda.
TARGET_COLOR = 'RED'

if SEND_ONE_TARGET:
    import time
    import json
    from pathlib import Path
    from icam_color_runtime import CamNaviSource, ColorModel, PlaneCalibration, V4Serial
    if not CALIBRATION_VERIFIED:
        raise RuntimeError('Selesaikan kalibrasi fisik dan isi corners_px dahulu')
    CALIBRATION = json.loads((Path(DEPLOY_DIR)/'camera_calibration.json').read_text())
    calibration = PlaneCalibration(CALIBRATION)
    detector = ColorModel(Path(DEPLOY_DIR)/'best.onnx')
    target_confidence = 0.60
    camera = CamNaviSource(CAMERA_DEVICE_NAME, *CALIBRATION['frame_size_wh'])
    robot = None
    try:
        robot = V4Serial(SERIAL_PORT, area_mm=calibration.area, dry_run=False)
        frame = camera.read()
        captured_at = time.monotonic()
        candidates = []
        for detection in detector.predict(frame, target_confidence):
            if detection['color'] != TARGET_COLOR or not detection['center_inside_mask']:
                continue
            xy = calibration.xy(detection['center_px'], frame.shape)
            if xy is not None:
                candidates.append((detection['confidence'], xy))
        if not candidates:
            raise RuntimeError('Tidak ada target warna yang valid dalam area')
        if time.monotonic()-captured_at > 2.0:
            raise RuntimeError('Hasil frame terlalu lama untuk perintah gerak; ukur latency perangkat')
        _, target_xy = max(candidates, key=lambda pair: pair[0])
        print('Target:', TARGET_COLOR, target_xy)
        robot.move_xy(*target_xy)
    finally:
        if robot is not None:
            robot.close()
        camera.close()

## 7. Buat bundle deployment

Bundle memuat model PT/ONNX, labels, audit dataset, metrik evaluasi, versi paket training,
script runtime, contoh kalibrasi, dan panduan ICAM. YAML training memakai path absolut
mesin training dan tidak diperlukan runtime. ZIP dataset asli tetap utuh.

In [ ]:
import importlib.metadata
from datetime import datetime, timezone

BUNDLE = ARTIFACT_ROOT/('deploy_' + RUN_DIR.name)
BUNDLE.mkdir(exist_ok=True)
shutil.copy2(BEST_PT, BUNDLE/'best.pt')
shutil.copy2(ONNX_PATH, BUNDLE/'best.onnx')
shutil.copy2(ROOT/'icam_color_runtime.py', BUNDLE/'icam_color_runtime.py')
(BUNDLE/'labels.txt').write_text('\n'.join(NAMES[i] for i in sorted(NAMES))+'\n')
(BUNDLE/'camera_calibration.example.json').write_text(json.dumps(CALIBRATION, indent=2))
if calibration is not None:
    shutil.copy2(WORK/'camera_calibration.json', BUNDLE/'camera_calibration.json')
manifest = dict(created_utc=datetime.now(timezone.utc).isoformat(), dataset_sha256=ZIP_SHA256,
                model=MODEL, names=NAMES, imgsz=IMGSZ, task='segment',
                preprocessing='BGR to RGB, stretch resize, NCHW float32 /255',
                onnx=dict(opset=13, batch=1, dynamic=False, nms=False),
                runtime='ONNX Runtime CPU; NPU integration not configured',
                training_versions={p: importlib.metadata.version(p) for p in
                                   ['ultralytics', 'torch', 'numpy', 'opencv-python', 'onnx', 'onnxruntime']},
                dataset_audit=REPORT, evaluation=VALIDATION)
(BUNDLE/'manifest.json').write_text(json.dumps(manifest, indent=2))
# ICAM Python 3.10 memakai NumPy 1.x untuk kompatibilitas OpenCV bawaan BSP.
# Requirements ini khusus ICAM, bukan cell instalasi training Colab Python 3.13.
(BUNDLE/'requirements-runtime.txt').write_text('numpy<2\nonnxruntime==1.20.1\npyserial==3.5\n')
readme = """ICAM-300 / Qualcomm QCS6490 / v4.ino
1. Gunakan Python 3.10 + CAMNavi2 + OpenCV dari BSP Advantech.
   Jika memakai venv, buat python3 -m venv --system-site-packages .venv
   Pastikan import CamNavi2, cv2, numpy berhasil sebelum menjalankan runtime.
2. pip install -r requirements-runtime.txt di environment yang kompatibel BSP.
   File ini khusus ICAM Python 3.10/NumPy 1.x, bukan environment training Colab.
   Wheel ONNX Runtime aarch64 harus cocok dengan Python/glibc Yocto Anda.
   Jika wheel tidak tersedia, gunakan build ONNX Runtime dari BSP/vendor atau
   jalankan runtime di PC dengan file/stream kamera yang dikonfigurasi.
3. Untuk CAMNavi2, lepaskan kontrol kamera dari web utility:
   sudo systemctl stop web.service
   Setelah sesi SDK ditutup: sudo systemctl start web.service
4. Uji gambar: python icam_color_runtime.py --model best.onnx --source meja.jpg
5. Uji kamera: python icam_color_runtime.py --source camnavi --device-name iCam500 --frames 100
   Nama device dan resolusi harus sesuai enum/kapabilitas SDK, bukan asumsi nama model kamera.
   Tambah --show hanya jika OpenCV GUI/display tersedia. Default headless, output JSON stdout.
   Camera timeout: periksa web.service, resolusi, pipeline JPEG/color, dan contoh CAMNavi2 vendor.
6. Isi camera_calibration.example.json sesuai urutan SAVE:1..4, simpan camera_calibration.json.
   python icam_color_runtime.py --calibration camera_calibration.json --frames 100
   CLI hanya mencetak command_preview. Tidak menggerakkan robot.
7. Untuk uji satu gerakan gunakan V4Serial dari script ini sesuai cell notebook.
   v4.ino harus sudah terpasang dan robot idle; tutup Serial Monitor lain.
   ZERO dan SAVE empat titik dilakukan manual pada pose fisik yang benar.
   AREA firmware harus sama dengan area_mm; XY tidak mengatur servo atau ketinggian.
8. ONNX ini FP32 CPU. QNN/SNPE/NPU memerlukan konversi terpisah, dukungan operator,
   decoder segmentation yang cocok, dan uji akurasi ulang setelah quantization.
9. Dataset asli didominasi crop tunggal. Nilai mAP bukan bukti deteksi multiobjek meja.
   Tambahkan frame penuh, contoh negatif, dan evaluasi rekaman ICAM sebelum penggunaan robot.
"""
(BUNDLE/'README-ICAM.txt').write_text(readme, encoding='utf-8')
BUNDLE_ZIP = Path(shutil.make_archive(str(BUNDLE), 'zip', BUNDLE))
print('Bundle:', BUNDLE_ZIP)
display(__import__('IPython').display.FileLink(str(BUNDLE_ZIP)))

In [ ]:
# Opsional: unduh bundle dari Colab setelah training/ekspor selesai.
DOWNLOAD_BUNDLE = False
if DOWNLOAD_BUNDLE:
    from google.colab import files
    files.download(str(BUNDLE_ZIP))

## 8. Menjalankan hasil di ICAM-300

Ekstrak bundle dan ikuti `README-ICAM.txt`. Training tidak perlu diulang di kamera.
Periksa import CAMNavi2/OpenCV bawaan BSP; pasang ONNX Runtime yang sesuai Python/aarch64/glibc.
Panduan lokal menyebut Yocto Linux + Python 3.10. Dukungan wheel Python bergantung image BSP.
Jika tidak cocok, gunakan build vendor atau jalankan runtime pada PC yang menerima file/stream ICAM.
Jangan mengasumsikan ICAM muncul sebagai webcam USB index 0.

```bash
# Hanya di ICAM, sesuai Programming Guide bab 3; hentikan pemilik kamera sebelumnya.
sudo systemctl stop web.service
python3 icam_color_runtime.py --source camnavi --device-name iCam500 --frames 100
# Untuk pratinjau GUI lokal, tambah --show; mode default headless menghasilkan JSON.
# Setelah runtime ditutup, bila ingin memakai web utility kembali:
sudo systemctl start web.service
```

Panduan vendor menggunakan nama device lama `iCam500` pada contoh ICAM-300.
Runtime menampilkan hasil enumerasi; sesuaikan `--device-name` dan resolusi dengan SDK sebenarnya.
Ia memakai callback JPEG warna, satu frame terbaru, tanpa antrean frame lama.
Eksposur/white balance/pencahayaan harus konsisten dengan data training. Jangan grayscale frame.

Uji ketiga kelas bersamaan, beberapa benda dengan warna sama, meja kosong, tangan/robot dalam gambar,
bayangan, dan pantulan. Catat false positive, objek terlewat, error pusat XY (mm), serta latency
di ICAM. Tidak ada angka FPS/akurasi hardware yang dijanjikan notebook ini.

Referensi:

- [Google Colab FAQ: runtime, GPU, dan penyimpanan Drive](https://research.google.com/colaboratory/faq.html)
- [Wheel ONNX 1.18.0 untuk Python 3.13](https://pypi.org/project/onnx/1.18.0/#files), [NumPy 2.2.6](https://pypi.org/project/numpy/2.2.6/#files)
- [Advantech ICAM-300: Qualcomm QCS6490](https://www.advantech.com/en-sg/products/ce666c81-b9fc-4675-b7aa-0c16ce758636/icam-300/mod_c348d281-94f9-4df7-82cb-cd9e35388c74)
- `Programming Guide for ICAM-300.pdf` lokal: tabel 1.1, bab 3, contoh CAMNavi2 bab 5 (hal. 54–56).
- `ICAM-300_User_Manual_Ed.2.pdf` lokal dan `v4.ino` sebagai acuan perangkat/protokol.
- [BCN3D Moveo](https://github.com/BCN3D/BCN3D-Moveo)
- [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11/), [segmentation](https://docs.ultralytics.com/tasks/segment/), [ONNX](https://docs.ultralytics.com/integrations/onnx/)
- [Konfigurasi Ultralytics v8.3.221 yang dipakai](https://github.com/ultralytics/ultralytics/blob/v8.3.221/ultralytics/cfg/default.yaml)
- [Qualcomm QIM pipeline dan postprocessing](https://www.qualcomm.com/developer/blog/2024/07/qualcomm-linux-sample-apps-ai-object-detection-parallel-ai)
- [Dataset Roboflow v5, CC BY 4.0](https://universe.roboflow.com/ahmad-zuhril-fahrizal/arm-warna-3/dataset/5)